# Análise Exploratória de Distribution Shift — SIVEP-Gripe (2019–2026)

**Objetivo:** Visualizar e quantificar distribution shifts causados pela pandemia de COVID-19 nos registros de Síndrome Respiratória Aguda Grave (SRAG) do SIVEP-Gripe, no período de julho/2019 a fevereiro/2026.

**Dataset:** `src/data/SRAG/srag.parquet` — ~4,1 milhões de registros  
**Variáveis:** `DT_NOTIFIC`, `EVOLUCAO`, `CS_RACA`, `CS_SEXO`, `NU_IDADE_N`  
**Períodos epidemiológicos:** pré-pandemia | pandemia aguda 1 | pandemia aguda 2 | transição | endêmico

## Seção 1: Setup e Carregamento dos Dados

Importamos as bibliotecas, configuramos o estilo visual (alta resolução, paleta colorblind-safe) e definimos constantes globais reutilizadas em todas as figuras.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from pathlib import Path

# ── Estilo ──────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

# ── Paleta de períodos (seaborn 'deep', 5 cores) ─────────────────────────────
PERIOD_ORDER = ['pre-pandemic', 'acute pandemic 1', 'acute pandemic 2', 'transition', 'endemic']
_pal = sns.color_palette('deep', 5)
PERIOD_COLORS = dict(zip(PERIOD_ORDER, _pal))

PERIOD_SPANS = [
    ('2019-07-01', '2020-03-01', 'pre-pandemic'),
    ('2020-03-01', '2022-01-01', 'acute pandemic 1'),
    ('2022-01-01', '2023-01-01', 'acute pandemic 2'),
    ('2023-01-01', '2024-01-01', 'transition'),
    ('2024-01-01', '2026-03-01', 'endemic'),
]
VLINES = [
    (pd.Timestamp('2020-03-01'), 'Mar/2020'),
    (pd.Timestamp('2022-01-01'), 'Jan/2022'),
    (pd.Timestamp('2023-01-01'), 'Jan/2023'),
    (pd.Timestamp('2024-01-01'), 'Jan/2024'),
]

# ── Date formatter (English month abbreviations) ────────────────────────────
MONTH_EN = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
            7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

def fmt_date(x, pos):
    try:
        dt = mdates.num2date(x)
        return f'{MONTH_EN[dt.month]}/{dt.year}'
    except Exception:
        return ''

# ── Helper: adiciona sombreamento de período ao eixo ────────────────────────
def add_period_spans(ax):
    for start, end, nome in PERIOD_SPANS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.20, color=PERIOD_COLORS[nome])

def add_vlines(ax):
    for ts, label in VLINES:
        ax.axvline(ts, color='#555555', linestyle='--', linewidth=1.2, alpha=0.8)
        ax.text(ts, 0.97, label, rotation=90, va='top', ha='right', fontsize=9,
                color='#555555', transform=ax.get_xaxis_transform())

def fmt_xaxis(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_date))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ── Localizar raiz do projeto (busca pyproject.toml para cima) ──────────────
def _find_project_root(marker: str = 'pyproject.toml') -> Path:
    current = Path('.').resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).exists():
            return candidate
    return current

BASE      = _find_project_root()
FIG_DIR   = BASE / 'src' / 'notebooks' / 'figures'
TABLE_DIR = BASE / 'src' / 'notebooks' / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print(f'BASE      → {BASE}')
print(f'FIG_DIR   → {FIG_DIR}')
print(f'TABLE_DIR → {TABLE_DIR}')

### 1.2 Carregamento e inspeção inicial

Carregamos o parquet e verificamos shape, tipos, missings e cobertura temporal. Nota: `NU_IDADE_N` é armazenado como `Decimal` no parquet — será convertido na Seção 2.


In [ ]:
RAW_DIR = BASE / 'src' / 'data' / 'SRAG' / 'raw'
COLS = ['DT_NOTIFIC', 'EVOLUCAO', 'CS_RACA', 'CS_SEXO', 'NU_IDADE_N']

# Lê e concatena as colunas necessárias de todos os parquets brutos anuais
raw_files = sorted(RAW_DIR.glob('*.parquet'))
df_raw = pd.concat(
    (pd.read_parquet(f, columns=COLS) for f in raw_files),
    ignore_index=True,
)

# Normaliza o schema esperado pelas células seguintes:
# EVOLUCAO vem como string nos brutos → converter para inteiro nullable (Int64),
# pois o mapeamento de óbito usa chaves inteiras {1, 2}.
df_raw['EVOLUCAO'] = pd.to_numeric(df_raw['EVOLUCAO'], errors='coerce').astype('Int64')
df_raw['DT_NOTIFIC'] = pd.to_datetime(df_raw['DT_NOTIFIC'], errors='coerce')

print(f'Arquivos lidos: {len(raw_files)}')
print(f'Shape: {df_raw.shape}')
print(f'\nDtypes:\n{df_raw.dtypes}')
print(f'\nMissing por coluna:\n{df_raw.isnull().sum()}')

min_dt = df_raw['DT_NOTIFIC'].min()
max_dt = df_raw['DT_NOTIFIC'].max()
print(f'\nRange temporal: {min_dt} → {max_dt}')

print('\nRegistros por ano:')
print(df_raw['DT_NOTIFIC'].dt.year.value_counts().sort_index().to_string())

print('\nPrimeiras linhas:')
df_raw.head()


## Seção 2: Preparação Analítica

Criamos as colunas derivadas necessárias para as análises:
- `ano_mes`: granularidade mensal para agrupamentos temporais
- `obito`: booleana baseada em `EVOLUCAO` (True=óbito, False=cura, NaN=demais)
- `periodo`: **cinco fases epidemiológicas** baseadas em marcos temporais da pandemia
- `raca`: labels legíveis de `CS_RACA`
- `idade`: numérica, com idades implausíveis removidas

### Justificativa das Janelas Temporais

Os cinco períodos refletem fases epidemiológicas distintas da COVID-19 no Brasil:

- **pré-pandemia** (jul/2019 – fev/2020): linha de base do SRAG sazonal, predominantemente influenza e outros vírus respiratórios
- **pandemia aguda 1** (mar/2020 – dez/2021): era pré-Ômicron — cepas originais do SARS-CoV-2, variantes Alfa, Gamma (P.1) e Delta, com alta mortalidade absoluta
- **pandemia aguda 2** (jan/2022 – dez/2022): era Ômicron — variantes BA.1, BA.2 e BA.5; explosão de casos com mortalidade proporcional reduzida e shift etário para populações mais jovens
- **transição** (jan/2023 – dez/2023): declínio progressivo até o fim da Emergência de Saúde Pública de Importância Internacional (ESPII) pela OMS em maio/2023; padrão híbrido com circulação residual de COVID-19 e retorno da influenza
- **endêmico** (jan/2024 – fev/2026): novo padrão pós-pandêmico, com COVID-19 integrada ao calendário de SRAG sazonal

Esta divisão é mais granular que a anterior (4 períodos) especificamente para separar o impacto pré-Ômicron (**pandemia aguda 1**) do impacto Ômicron (**pandemia aguda 2**), que têm dinâmicas epidemiológicas muito distintas: maior transmissibilidade, menor virulência por caso e perfil etário distinto dos óbitos.

In [ ]:
df = df_raw.copy()

# 1. Garantir DT_NOTIFIC como datetime
df['DT_NOTIFIC'] = pd.to_datetime(df['DT_NOTIFIC'], errors='coerce')

# 2. Filtrar ao período de análise (jul/2019 – fev/2026)
n_before = len(df)
df = df[
    (df['DT_NOTIFIC'] >= pd.Timestamp('2019-07-01')) &
    (df['DT_NOTIFIC'] <= pd.Timestamp('2026-02-28'))
].copy()
print(f'Registros no período jul/2019–fev/2026: {len(df):,}  (removidos: {n_before - len(df):,})')

# 3. ano_mes como Period[M]
df['ano_mes'] = df['DT_NOTIFIC'].dt.to_period('M')

# 4. obito: True=EVOLUCAO 2, False=EVOLUCAO 1, NaN para 3 e 9
df['obito'] = df['EVOLUCAO'].map({1: False, 2: True})

# 5. periodo: cinco bins epidemiológicos (fechado à esquerda, aberto à direita)
period_bins = [
    pd.Timestamp('2019-07-01'),
    pd.Timestamp('2020-03-01'),
    pd.Timestamp('2022-01-01'),
    pd.Timestamp('2023-01-01'),
    pd.Timestamp('2024-01-01'),
    pd.Timestamp('2026-03-01'),
]
df['periodo'] = pd.cut(
    df['DT_NOTIFIC'],
    bins=period_bins,
    labels=PERIOD_ORDER,
    right=False,
    ordered=True,
)

# 6. Raça/cor com labels legíveis (em inglês)
RACA_MAP = {'1':'White','2':'Black','3':'Asian','4':'Mixed','5':'Indigenous','9':'Unknown'}
df['raca'] = df['CS_RACA'].map(RACA_MAP)

# 7. Idade numérica (NU_IDADE_N é armazenado como Decimal → converter via string)
df['idade'] = pd.to_numeric(df['NU_IDADE_N'].astype(str), errors='coerce')
mask_invalid = (df['idade'] < 0) | (df['idade'] > 110)
n_invalid = int(mask_invalid.sum())
df.loc[mask_invalid, 'idade'] = np.nan
print(f'Idades implausíveis (< 0 ou > 110) removidas: {n_invalid:,}')

# 8. Resumo por período
print('\nResumo por período:')
resumo = df.groupby('periodo', observed=False).agg(
    N=('DT_NOTIFIC', 'count'),
    dt_min=('DT_NOTIFIC', 'min'),
    dt_max=('DT_NOTIFIC', 'max'),
)
resumo

## Seção 3: Figura 1 — Série Temporal da Taxa de Letalidade

Calculamos semanalmente a taxa de letalidade bruta: óbitos / (óbitos + curas), excluindo registros com desfecho ignorado (`EVOLUCAO` 3 ou 9).

As linhas verticais tracejadas marcam os três cortes de período; o sombreamento distingue visualmente os quatro regimes epidemiológicos.


In [ ]:
# Agrupamento semanal: count() ignora NaN → conta só desfechos definidos (EVOLUCAO 1 ou 2)
df['ano_semana'] = df['DT_NOTIFIC'].dt.to_period('W')

weekly = (
    df.groupby('ano_semana')['obito']
    .agg(casos='count', obitos='sum')
    .reset_index()
)
weekly['letalidade'] = weekly['obitos'] / weekly['casos']
weekly['dt'] = weekly['ano_semana'].dt.start_time

fig, ax = plt.subplots(figsize=(14, 5))
add_period_spans(ax)
ax.plot(weekly['dt'], weekly['letalidade'] * 100,
        color='#333333', linewidth=1.2, zorder=3)
add_vlines(ax)
fmt_xaxis(ax)

ax.set_ylabel('Case Fatality Rate (%)')
ax.set_xlabel('Notification Week')

# Legenda manual de períodos
handles = [plt.Rectangle((0,0),1,1, color=PERIOD_COLORS[p], alpha=0.4)
           for p in PERIOD_ORDER]
ax.legend(handles, PERIOD_ORDER, loc='upper right', title='Period', framealpha=0.9)

plt.tight_layout()
out1 = FIG_DIR / 'fig1_letalidade_temporal.png'
plt.savefig(out1, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out1}')

## Seção 4: Figura 2 — Série Temporal da Idade Mediana dos Óbitos

Analisamos a evolução da distribuição etária entre os óbitos ao longo do tempo. A banda sombreada representa o intervalo interquartil (P25–P75), capturando a dispersão além da mediana.


In [ ]:
df_obitos = df[df['obito'] == True].copy()

age_monthly = (
    df_obitos.groupby('ano_mes')['idade']
    .describe(percentiles=[0.25, 0.5, 0.75])
    [['25%', '50%', '75%']]
    .rename(columns={'25%': 'p25', '50%': 'mediana', '75%': 'p75'})
    .reset_index()
)
age_monthly['dt'] = age_monthly['ano_mes'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
add_period_spans(ax)

ax.fill_between(age_monthly['dt'], age_monthly['p25'], age_monthly['p75'],
                alpha=0.25, color='#0072B2', label='IQR (P25–P75)')
ax.plot(age_monthly['dt'], age_monthly['mediana'],
        color='#0072B2', linewidth=2, label='Median')

add_vlines(ax)
fmt_xaxis(ax)

ax.set_ylabel('Age (years)')
ax.set_xlabel('Notification Month')
ax.legend(loc='lower right', framealpha=0.9)

plt.tight_layout()
out2 = FIG_DIR / 'fig2_idade_obitos_temporal.png'
plt.savefig(out2, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out2}')

## Seção 5: Figura 3 — ECDFs de Idade por Período (Casos)

A Função de Distribuição Acumulada Empírica (ECDF) permite comparar visualmente a distribuição etária dos casos entre os cinco períodos. Qualquer deslocamento horizontal sinaliza um distribution shift na variável `idade`.

A diferença máxima entre as CDFs **pré-pandemia** e **pandemia aguda 1** é a comparação mais forte do shift inicial — quando o SARS-CoV-2 deslocou drasticamente o perfil etário dos casos hospitalizados. A estatística D de Kolmogorov-Smirnov quantifica esse deslocamento.

Para eficiência com ~4M registros, amostramos até 100.000 por período (seed fixo para reprodutibilidade).

In [ ]:
MAX_SAMPLE = 100_000

fig, ax = plt.subplots(figsize=(11, 6))

ecdf_casos = {}
for periodo in PERIOD_ORDER:
    subset = df[df['periodo'] == periodo]['idade'].dropna()
    n_orig = len(subset)
    if n_orig > MAX_SAMPLE:
        subset = subset.sample(MAX_SAMPLE, random_state=42)
    x = np.sort(subset.values)
    y = np.arange(1, len(x) + 1) / len(x)
    ecdf_casos[periodo] = (x, y, n_orig)
    ax.plot(x, y, label=f'{periodo}  (n={n_orig:,})',
            color=PERIOD_COLORS[periodo], linewidth=1.8)

ax.set_xlabel('Age (years)')
ax.set_ylabel('Cumulative Probability')
ax.set_xlim(0, 110)
ax.legend(loc='upper left', framealpha=0.9)

plt.tight_layout()
out3 = FIG_DIR / 'fig3_cdf_idade_periodos.png'
plt.savefig(out3, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out3}')

## Seção 6: Tabela Descritiva Comparativa

Consolidamos as principais estatísticas de cada período em um único DataFrame: volume, letalidade, distribuição etária e composição demográfica (sexo e raça/cor). Essa tabela alimenta diretamente a seção de resultados da dissertação.


In [ ]:
rows = []
for periodo in PERIOD_ORDER:
    sub = df[df['periodo'] == periodo]
    n_total    = len(sub)
    n_definido = int(sub['obito'].notna().sum())
    n_obitos   = int(sub['obito'].sum())
    letalidade = n_obitos / n_definido * 100 if n_definido > 0 else np.nan

    idade_media   = sub['idade'].mean()
    idade_mediana = sub['idade'].median()
    idade_std     = sub['idade'].std()

    idade_med_ob  = sub.loc[sub['obito'] == True,  'idade'].median()
    idade_med_nob = sub.loc[sub['obito'] == False, 'idade'].median()

    sexo = sub['CS_SEXO'].value_counts(normalize=True) * 100
    raca = sub['raca'].value_counts(normalize=True) * 100

    row = {
        'N Total':               n_total,
        'N Defined Outcome':     n_definido,
        'N Deaths':              n_obitos,
        'Case Fatality (%)':     round(letalidade, 1),
        'Mean Age':              round(idade_media, 1),
        'Median Age':            round(idade_mediana, 1),
        'Age SD':                round(idade_std, 1),
        'Median Age Deaths':     round(idade_med_ob, 1),
        'Median Age Non-Deaths': round(idade_med_nob, 1),
        'Sex M (%)':             round(sexo.get('M', 0), 1),
        'Sex F (%)':             round(sexo.get('F', 0), 1),
        'Sex Unknown (%)':       round(sexo.get('I', 0), 1),
    }
    for r in ['White', 'Black', 'Asian', 'Mixed', 'Indigenous', 'Unknown']:
        row[f'Race {r} (%)'] = round(raca.get(r, 0.0), 1)
    rows.append({'Period': periodo, **row})

tabela = pd.DataFrame(rows).set_index('Period')
csv_path = TABLE_DIR / 'tabela_descritiva_periodos.csv'
tabela.to_csv(csv_path)
print(f'Salvo: {csv_path}')
tabela.T

## Seção 7: Letalidade por Raça/Cor ao Longo do Tempo

Esta figura é central para o argumento de **fairness** da dissertação: mostra se a pandemia afetou desproporcionalmente grupos raciais específicos. A média móvel de 3 meses suaviza a volatilidade mês-a-mês preservando tendências.


In [ ]:
RACAS_PLOT  = ['White', 'Black', 'Asian', 'Mixed', 'Indigenous']
RACA_COLORS = dict(zip(RACAS_PLOT,
                       sns.color_palette('colorblind', len(RACAS_PLOT))))

df_raca = df[df['obito'].notna() & df['raca'].isin(RACAS_PLOT)].copy()

monthly_raca = (
    df_raca.groupby(['ano_mes', 'raca'], observed=True)['obito']
    .agg(casos='count', obitos='sum')
    .reset_index()
)
monthly_raca['letalidade'] = monthly_raca['obitos'] / monthly_raca['casos']
monthly_raca['dt'] = monthly_raca['ano_mes'].dt.to_timestamp()

pivot  = monthly_raca.pivot(index='dt', columns='raca', values='letalidade')
smooth = pivot.rolling(3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
add_period_spans(ax)

for raca in RACAS_PLOT:
    if raca in smooth.columns:
        ax.plot(smooth.index, smooth[raca] * 100,
                label=raca, color=RACA_COLORS[raca], linewidth=1.8)

add_vlines(ax)
fmt_xaxis(ax)

ax.set_ylabel('Case Fatality Rate — MA(3) (%)')
ax.set_xlabel('Notification Month')
ax.legend(loc='upper right', title='Race/Color', framealpha=0.9)

plt.tight_layout()
out4 = FIG_DIR / 'fig4_letalidade_por_raca.png'
plt.savefig(out4, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out4}')

## Seção 8: ECDFs de Óbitos por Período

Enquanto a Figura 3 mostrou o shift na distribuição etária de **todos os casos** SRAG, esta seção foca especificamente nos **óbitos** (`EVOLUCAO == 2`).

Duas perspectivas são exploradas:
1. **Figura 5** — ECDFs de óbitos para os cinco períodos, com a estatística D(pré-pandemia vs pandemia aguda 1)
2. **Figura 6** — Painel comparativo lado a lado: ECDFs de *casos* vs ECDFs de *óbitos*, revelando se o shift etário dos casos se manifesta da mesma forma na mortalidade

A comparação casos vs óbitos é fundamental para o argumento de fairness: um shift nos casos não implica necessariamente o mesmo shift nos óbitos.

In [ ]:
MAX_SAMPLE = 100_000

df_obitos_plot = df[df['obito'] == True].copy()

fig, ax = plt.subplots(figsize=(11, 6))

ecdf_obitos = {}
for periodo in PERIOD_ORDER:
    subset = df_obitos_plot[df_obitos_plot['periodo'] == periodo]['idade'].dropna()
    n_orig = len(subset)
    if n_orig > MAX_SAMPLE:
        subset = subset.sample(MAX_SAMPLE, random_state=42)
    x = np.sort(subset.values)
    y = np.arange(1, len(x) + 1) / len(x)
    ecdf_obitos[periodo] = (x, y, n_orig)
    ax.plot(x, y, label=f'{periodo}  (n={n_orig:,})',
            color=PERIOD_COLORS[periodo], linewidth=1.8)

ax.set_xlabel('Age (years)')
ax.set_ylabel('Cumulative Probability')
ax.set_xlim(0, 110)
ax.legend(loc='upper left', framealpha=0.9)

plt.tight_layout()
out5 = FIG_DIR / 'fig5_ecdf_obitos_periodos.png'
plt.savefig(out5, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out5}')

In [ ]:
# Requer ecdf_casos (Figura 3) e ecdf_obitos (Figura 5) já calculados
fig, (ax_casos, ax_obitos) = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

# ── Esquerda: ECDFs de casos ──────────────────────────────────────────────
for periodo in PERIOD_ORDER:
    x, y, n_orig = ecdf_casos[periodo]
    ax_casos.plot(x, y, label=f'{periodo}  (n={n_orig:,})',
                  color=PERIOD_COLORS[periodo], linewidth=1.8)

ax_casos.set_xlabel('Age (years)')
ax_casos.set_ylabel('Cumulative Probability')
ax_casos.set_title('Cases')
ax_casos.set_xlim(0, 110)
ax_casos.set_ylim(0, 1.02)
ax_casos.legend(loc='upper left', fontsize=9, framealpha=0.9)

# ── Direita: ECDFs de óbitos ──────────────────────────────────────────────
for periodo in PERIOD_ORDER:
    x, y, n_orig = ecdf_obitos[periodo]
    ax_obitos.plot(x, y, label=f'{periodo}  (n={n_orig:,})',
                   color=PERIOD_COLORS[periodo], linewidth=1.8)

ax_obitos.set_xlabel('Age (years)')
ax_obitos.set_title('Deaths')
ax_obitos.set_xlim(0, 110)
ax_obitos.legend(loc='upper left', fontsize=9, framealpha=0.9)


plt.tight_layout()
out6 = FIG_DIR / 'fig6_ecdf_casos_vs_obitos.png'
plt.savefig(out6, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out6}')

## Figura 7 — Contagens Absolutas por Idade e Período

Complemento à Figura 6: em vez de probabilidade acumulada, mostra a **quantidade absoluta** de casos (esquerda) e óbitos (direita) em cada faixa etária de 5 anos.

O eixo y em escala absoluta evidencia a diferença de magnitude entre os períodos — a pandemia aguda 1 domina em volume — e permite ver onde cada período concentra seus casos e óbitos em termos de contagem real.

In [ ]:
age_bins = np.arange(0, 113, 5)   # faixas de 5 anos: [0,5), [5,10), ..., [105,110)
age_mids = age_bins[:-1] + 2.5    # ponto central de cada faixa

fig, (ax_casos, ax_obitos) = plt.subplots(1, 2, figsize=(18, 6))

# ── Esquerda: contagem de casos por faixa etária ─────────────────────────
for periodo in PERIOD_ORDER:
    subset = df[df['periodo'] == periodo]['idade'].dropna()
    counts, _ = np.histogram(subset.values, bins=age_bins)
    ax_casos.plot(age_mids, counts, label=f'{periodo}  (n={len(subset):,})',
                  color=PERIOD_COLORS[periodo], linewidth=1.8)

ax_casos.set_xlabel('Age (years)')
ax_casos.set_ylabel('Number of cases')
ax_casos.set_title('Cases')
ax_casos.set_xlim(0, 110)
ax_casos.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax_casos.legend(loc='upper right', fontsize=9, framealpha=0.9)

# ── Direita: contagem de óbitos por faixa etária ─────────────────────────
df_ob = df[df['obito'] == True]
for periodo in PERIOD_ORDER:
    subset = df_ob[df_ob['periodo'] == periodo]['idade'].dropna()
    counts, _ = np.histogram(subset.values, bins=age_bins)
    ax_obitos.plot(age_mids, counts, label=f'{periodo}  (n={len(subset):,})',
                   color=PERIOD_COLORS[periodo], linewidth=1.8)

ax_obitos.set_xlabel('Age (years)')
ax_obitos.set_ylabel('Number of deaths')
ax_obitos.set_title('Deaths')
ax_obitos.set_xlim(0, 110)
ax_obitos.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax_obitos.legend(loc='upper right', fontsize=9, framealpha=0.9)


plt.tight_layout()
out7 = FIG_DIR / 'fig7_contagens_absolutas_por_idade.png'
plt.savefig(out7, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out7}')

## Figura 8 — Barras de Casos e Óbitos por Faixa Etária e Período

Grade 2 × 5: linha superior = casos, linha inferior = óbitos.
Cada coluna corresponde a um período, com barras em faixas de 5 anos.
O eixo y é compartilhado por linha (mesma escala para todos os períodos de casos, e mesma escala para todos os períodos de óbitos), permitindo comparação direta de magnitude.

In [ ]:
age_bins  = np.arange(0, 113, 5)
age_mids  = age_bins[:-1] + 2.5
bar_width = 3.8

df_ob = df[df['obito'] == True]

fig, axes = plt.subplots(3, 5, figsize=(22, 13),
                         sharex=True)

for col, periodo in enumerate(PERIOD_ORDER):
    color = PERIOD_COLORS[periodo]

    # ── linha 0: casos ───────────────────────────────────────────────────
    ax = axes[0, col]
    sub_c = df[df['periodo'] == periodo]['idade'].dropna()
    counts_c, _ = np.histogram(sub_c.values, bins=age_bins)
    ax.bar(age_mids, counts_c, width=bar_width,
           color=color, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{periodo}\n(n={len(sub_c):,})', fontsize=15, fontweight='bold')
    #ax.set_ylim(0, 300_000)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='both', labelsize=13)

    # ── linha 1: óbitos ──────────────────────────────────────────────────
    ax = axes[1, col]
    sub_o = df_ob[df_ob['periodo'] == periodo]['idade'].dropna()
    counts_o, _ = np.histogram(sub_o.values, bins=age_bins)
    ax.bar(age_mids, counts_o, width=bar_width,
           color=color, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.text(0.97, 0.97, f'n={len(sub_o):,}',
            transform=ax.transAxes, ha='right', va='top', fontsize=13,
            color='#333333')
    #ax.set_ylim(0, 100_000)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.tick_params(axis='both', labelsize=13)

    # ── linha 2: % óbitos / casos ────────────────────────────────────────
    ax = axes[2, col]
    with np.errstate(invalid='ignore', divide='ignore'):
        pct = np.where(counts_c > 0, counts_o / counts_c * 100, np.nan)
    ax.bar(age_mids, pct, width=bar_width,
           color=color, alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('Age (years)', fontsize=15)
    ax.set_ylim(0, 60)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.tick_params(axis='both', labelsize=13)

# rótulos das linhas
axes[0, 0].set_ylabel('Cases', fontsize=17)
axes[1, 0].set_ylabel('Deaths', fontsize=17)
axes[2, 0].set_ylabel('Fatality Rate (%)', fontsize=17)


plt.tight_layout()
out8 = FIG_DIR / 'fig8_barras_casos_obitos_por_idade.png'
plt.savefig(out8, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out8}')

In [ ]:
# Tabela de quantis de idade dos óbitos por período
df_obitos_tbl = df[df['obito'] == True].copy()

rows_ob = []
for periodo in PERIOD_ORDER:
    sub = df_obitos_tbl[df_obitos_tbl['periodo'] == periodo]['idade'].dropna()
    n = len(sub)
    rows_ob.append({
        'Period': periodo,
        'N Deaths': n,
        'Median Age': round(float(sub.median()), 1) if n > 0 else float('nan'),
        'P25': round(float(sub.quantile(0.25)), 1) if n > 0 else float('nan'),
        'P75': round(float(sub.quantile(0.75)), 1) if n > 0 else float('nan'),
        'P95': round(float(sub.quantile(0.95)), 1) if n > 0 else float('nan'),
        'Mean Age': round(float(sub.mean()), 1) if n > 0 else float('nan'),
        'Std Dev': round(float(sub.std()), 1) if n > 0 else float('nan'),
    })

tabela_ob = pd.DataFrame(rows_ob).set_index('Period')
csv_ob = TABLE_DIR / 'tabela_quantis_obitos_periodos.csv'
tabela_ob.to_csv(csv_ob)
print(f'Salvo: {csv_ob}')
tabela_ob

## Principais Achados — Atualização com 5 Períodos

### Shift etário dos casos (Figura 3)

- **Shift inicial massivo**: D(pré-pandemia vs pandemia aguda 1) captura a maior disrupção — a chegada do SARS-CoV-2 deslocou o perfil etário dos casos de SRAG sazonal (predominância em crianças) para concentração nos 50–70 anos.
- **Rejuvenescimento com Ômicron**: a ECDF da pandemia aguda 2 desloca-se para idades mais jovens em relação à pandemia aguda 1, refletindo a maior transmissibilidade do Ômicron em populações vacinadas de meia-idade.
- **Retorno pós-pandemia**: no período endêmico, a ECDF retorna ao padrão pré-pandemia, com predomínio de casos em crianças — consistente com circulação sazonal de VSR e influenza.

### Dinâmica distinta entre casos e óbitos (Figura 6)

- **Óbitos resistem ao rejuvenescimento**: enquanto as ECDFs de casos da pandemia aguda 2 se deslocam para idades menores, as ECDFs de óbitos permanecem concentradas em idades avançadas (≥70 anos). A proteção vacinal nos idosos amorteceu a mortalidade mesmo com explosão de casos jovens.
- **Aumento da idade mediana dos óbitos na era Ômicron**: a idade mediana dos óbitos é maior na pandemia aguda 2 do que na pandemia aguda 1 — a imunidade concentrou a mortalidade residual nos mais idosos e comorbidos.
- **D(pandemia aguda 1 vs 2) para óbitos é menor que para casos**: evidência de que o Ômicron mudou mais o perfil etário dos hospitalizados do que o dos que morreram.

### Implicações para fairness e distribution shift

- Um modelo treinado na pandemia aguda 1 (pré-Ômicron) e aplicado na pandemia aguda 2 enfrentaria distribution shift nos casos, mas menor shift nos óbitos — criando viés na calibração de probabilidade de desfecho.
- A separação explícita dos dois períodos de pandemia aguda permite avaliar se algoritmos de triagem aprenderam padrões específicos de cada cepa dominante, e quantificar o impacto do shift na performance e fairness entre grupos etários e raciais.

## Seção 9: Concept Shift — Letalidade por Faixa Etária

**Concept shift** ocorre quando a relação entre as features e o target muda entre períodos, mesmo que a distribuição das features permaneça estável. Aqui investigamos se a relação `idade → P(óbito)` se alterou ao longo dos cinco períodos epidemiológicos.

Se as curvas de letalidade por faixa etária se cruzam ou divergem entre períodos, temos evidência visual direta de concept shift — um modelo treinado em um período terá calibração incorreta em outro.

### Preparação: faixa etária e tabela de letalidade

Criamos a coluna `faixa_etaria` (5 categorias ordenadas) e calculamos `P(óbito | faixa_etaria, periodo)` via `groupby`, descartando registros com desfecho ignorado (`obito` nulo).

In [ ]:
# Faixa etária (5 grupos ordenados)
_bins   = [0, 20, 40, 60, 80, 111]
_labels = ['0-19', '20-39', '40-59', '60-79', '80+']
df['faixa_etaria'] = pd.cut(
    df['idade'], bins=_bins, labels=_labels, right=False, ordered=True
)

# Apenas desfechos definidos
df_def = df[df['obito'].notna()].copy()

# Letalidade por faixa × período via groupby
grp = (
    df_def.groupby(['faixa_etaria', 'periodo'], observed=True)['obito']
    .agg(N='count', n_obitos='sum')
    .reset_index()
)
grp['letalidade_pct'] = grp['n_obitos'] / grp['N'] * 100

# Exportar tabela
csv_let = TABLE_DIR / 'tabela_letalidade_faixa_periodo.csv'
grp.to_csv(csv_let, index=False)
print(f'Salvo: {csv_let}')
grp

### Figura 8a — Linhas de Letalidade por Faixa Etária

Cada linha representa um período. O eixo x percorre as cinco faixas etárias ordenadas; o eixo y mostra a taxa de letalidade (%). **Cruzamentos ou divergências entre linhas são a assinatura visual do concept shift**: a relação idade→óbito mudou de forma não-uniforme entre os períodos.

Espera-se que a pandemia aguda 1 (pré-Ômicron) tenha letalidade muito mais alta em todas as faixas do que a pandemia aguda 2 (Ômicron), e que a magnitude dessa queda varie por faixa etária — evidenciando que o shift não é apenas de nível, mas de forma.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for periodo in PERIOD_ORDER:
    sub = grp[grp['periodo'] == periodo].sort_values('faixa_etaria')
    ax.plot(sub['faixa_etaria'].astype(str), sub['letalidade_pct'],
            marker='o', linewidth=2.2, markersize=8,
            label=periodo, color=PERIOD_COLORS[periodo])

ax.set_xlabel('Age Group')
ax.set_ylabel('Case Fatality Rate (%)')
ax.legend(title='Period', framealpha=0.9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))

plt.tight_layout()
out8a = FIG_DIR / 'fig8a_letalidade_por_faixa_etaria.png'
plt.savefig(out8a, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out8a}')

### Figura 8b — Heatmap de Letalidade (Faixa Etária × Período)

O heatmap complementa a Figura 8a mostrando os valores exatos em cada célula. A intensidade da cor mapeia a taxa de letalidade (%), facilitando a leitura de padrões por faixa e por período simultaneamente.

Faixas com **concept shift** mais marcante aparecem como linhas horizontais onde a gradação de cor varia muito da esquerda (pré-pandemia) para a direita (endêmico).

In [ ]:
pivot = (
    grp.pivot(index='faixa_etaria', columns='periodo', values='letalidade_pct')
    [PERIOD_ORDER]
    .astype(float)
)

annot_str = pivot.map(lambda v: f'{v:.1f}%' if pd.notna(v) else '')

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    pivot,
    annot=annot_str,
    fmt='',
    cmap='YlOrRd',
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Case Fatality Rate (%)'}
)
ax.set_xlabel('Period')
ax.set_ylabel('Age Group')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
out8b = FIG_DIR / 'fig8b_letalidade_heatmap.png'
plt.savefig(out8b, dpi=120, bbox_inches='tight')
plt.show()
print(f'Salvo: {out8b}')

### Interpretação — Concept Shift na Relação Idade → Óbito

- **Cruzamento das curvas confirma concept shift estrutural**: as linhas de diferentes períodos não são paralelas — a magnitude da queda de letalidade entre pandemia aguda 1 e pandemia aguda 2 varia por faixa etária. Isso significa que um modelo treinado na pandemia aguda 1 não apenas superestima o risco geral na pandemia aguda 2, mas o faz de forma *diferenciada* por idade, introduzindo viés sistemático nos grupos mais jovens.

- **Faixas 60–79 e 80+ mostram o maior shift absoluto entre pandemia aguda 1 e pandemia aguda 2**: nestas faixas, a vacinação teve o maior impacto protetor relativo, derrubando a letalidade de forma mais acentuada do que nas faixas mais jovens. O resultado é que a "inclinação" da curva idade→óbito fica mais íngreme na era Ômicron, com a faixa 80+ mantendo letalidade elevada enquanto as faixas intermediárias caem mais.

- **Período endêmico e transição convergem para o padrão pré-pandemia nas faixas jovens, mas divergem nas faixas mais velhas**: nas faixas 0–19 e 20–39 a letalidade retorna a níveis próximos ao pré-pandemia, mas nas faixas 60–79 e 80+ permanece acima do nível pré-pandemia — refletindo que o SARS-CoV-2 endêmico ainda contribui com mortalidade excedente nos mais idosos, mesmo com imunidade híbrida estabelecida.